# CNAME Analysis — CNAME-Based Tracker Cloaking

This notebook investigates **CNAME cloaking**, a technique where tracking domains are disguised behind first-party DNS aliases (CNAME records) to bypass ad blockers and tracker detection.


For each HTTP request, the CNAME of the request hostname is resolved and re-checked against tracker lists. This reveals trackers that evade URL-based detection by hiding behind a first-party subdomain. The notebook examines which sites use CNAME cloaking, which tracking companies are behind the aliases, and whether these trackers persist after cookie rejection.


## Setup and data load

In [ ]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR


sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False)

## Only analyze data from the tracking exp_mode

In [15]:
df = df[df.exp_mode == 'tracking']

### Detect CNAME-based evasion
- Find requests that are **not** flagged as trackers based on URL, but **are** flagged as trackers based on their CNAME
- `is_cname_tracker` is based on replacing the `req_hostname` with the CNAME and re-checking against tracker lists
- Example: `customer.etracker.com` (tracker) is the CNAME for `e.medikamente-per-klick.de` (not classified as tracker)

In [ ]:
df[~df.is_tracker & df.is_cname_tracker].drop_duplicates(subset=['site_hostname', 'cname'])[
    ['consent_mode', 'site_domain', 'req_hostname', 'cname',
     'url', 'is_cname_tracker', 'resource_type']]

### Summary: CNAME-based evasion

In [ ]:
cname_evasion = df[~df.is_tracker & df.is_cname_tracker].drop_duplicates(subset=['site_hostname', 'cname'])

n_evasion_requests = df[~df.is_tracker & df.is_cname_tracker].shape[0]
n_evasion_sites = cname_evasion['site_domain'].nunique()
n_evasion_cname_domains = cname_evasion['cname_domain'].nunique()
n_evasion_pairs = cname_evasion.shape[0]

print(f"Requests evading URL-based detection via CNAME: {n_evasion_requests:,}")
print(f"Unique site–CNAME pairs:  {n_evasion_pairs}")
print(f"Sites affected:           {n_evasion_sites}")
print(f"Distinct CNAME domains:   {n_evasion_cname_domains}")
print(f"\nCNAME domains used for evasion:")
print(cname_evasion['cname_domain'].value_counts().to_string())

### First-party CNAME aliases

In [ ]:
df[~df.third_party & (df.cname_domain != df.req_domain) & (df.cname != '')].\
    drop_duplicates(['site_domain', 'req_hostname', 'cname'])[
        ['consent_mode', 'site_domain', 'req_hostname', 'cname', 'is_tracker', 'is_cname_tracker']].sort_values(['is_tracker', 'is_cname_tracker'], ascending=False)

In [ ]:
df[~df.third_party & df.is_tracker & (df.cname_domain != df.req_domain) & (df.cname != '')].\
    drop_duplicates(['req_hostname', 'cname'])[['req_hostname', 'cname']]

### CNAME tracker entities
Which tracking companies are behind the CNAME aliases?

In [ ]:
fp_cname_trackers = df[~df.third_party & df.is_cname_tracker & (df.cname_domain != df.req_domain) & (df.cname != '')]

entity_summary = (
    fp_cname_trackers
    .drop_duplicates(['site_domain', 'cname_domain'])
    .groupby('cname_entity')
    .agg(
        sites=('site_domain', 'nunique'),
        cname_domains=('cname_domain', 'nunique'),
    )
    .sort_values('sites', ascending=False)
)
entity_summary

### CNAME-cloaked tracking by consent mode
Do CNAME-cloaked trackers persist after cookie rejection?

In [ ]:
cname_by_consent = (
    fp_cname_trackers
    .drop_duplicates(['consent_mode', 'site_domain', 'cname_domain'])
    .groupby('consent_mode')
    .agg(
        sites=('site_domain', 'nunique'),
        unique_cname_domains=('cname_domain', 'nunique'),
        total_pairs=('cname_domain', 'size'),
    )
)
print("First-party CNAME-cloaked tracker presence by consent mode:\n")
cname_by_consent